## Part 1: Data Cleaning & Validation

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display
# Path, by default the data.db is stored in Lecture4 Folder
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file()
)
db_path = repo_root / "Lectures" / "Lecture 4" / "data" / "data.db"
if not db_path.is_file():
    raise FileNotFoundError(f"Database not found: {db_path}")
print("Database:", db_path)

Database: /Users/wangyisu/Desktop/Documents/pre-courses/Python for Financial Engineers/UCB-MFE-python-preprogram-2026-winter/Lectures/Lecture 4/data/data.db


### 1.1 Load and Inspect the Data

In [2]:
# Load the homework table read-only and inspect its structure.
with sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn,parse_dates = ["ts"])
raw.info()
display(raw.describe())
print("the columns:",raw.columns.to_list())
print("Date range:", raw["ts"].min(), "to", raw["ts"].max())
# handing ticker mislabeled
raw["ticker"] = raw['ticker'].astype(str).str.extract(r'>([A-Z]+)</span>')[0].where(lambda x: x.notna(), raw['ticker'])
print("list of tickers:",raw["ticker"].unique().tolist())
# Remove exact duplicates from the analysis view before calculating daily returns.
cols = ["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume"]
exact_duplicate = raw.duplicated(keep="first")
df = raw.loc[~exact_duplicate].sort_values(["ticker", "ts"]).reset_index(drop=True).copy()
print("duplicate samples:", exact_duplicate.sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


the columns: ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']
Date range: 1962-01-02 00:00:00 to 2024-12-06 00:00:00
list of tickers: ['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS', 'CSCO', 'SHW', 'KO', 'VZ', 'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM', 'UNH', 'NKE', 'NVDA', 'AMGN', 'MCD', 'V', 'MRK', 'WMT', 'JPM', 'MSFT', 'AMZN']
duplicate samples: 29460


### 1.2 Detect Data Quality Issues

In [3]:
# Missing Data Issues
def detect_missing_cells(data):
    return data.isna().sum()
def detect_missing_dates(data, calendar):
    observed = data[["ticker", "ts"]].drop_duplicates()
    bounds = observed.groupby("ticker", as_index=False)["ts"].agg(start="min", end="max")
    expected = calendar[["ticker", "ts"]].drop_duplicates().merge(bounds, on="ticker")
    expected = expected.loc[expected["ts"].between(expected["start"], expected["end"]), ["ticker", "ts"]]
    merged = expected.merge(observed, on=["ticker", "ts"], how="left", indicator=True)
    return merged.loc[merged["_merge"].eq("left_only"), ["ticker", "ts"]].reset_index(drop=True)
with sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True) as conn:
    reference = pd.read_sql_query("SELECT ticker, ts FROM ohlc", conn, parse_dates=["ts"])
missing_dates = detect_missing_dates(df, reference)
display(detect_missing_cells(df).rename("missing_cells").to_frame())
print(f"Missing trading dates across {df['ticker'].nunique()} tickers: {len(missing_dates)}")

,missing_cells
ts,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0
ticker,0


Missing trading dates across 30 tickers: 0


In [4]:
# outliers
# Flag valid raw-close returns beyond three ticker-specific sample SDs and invalid prices
def detect_outliers(data):
    previous = data.groupby("ticker")["close"].shift(1)
    ret = (data["close"] - previous) / previous
    valid = np.isfinite(data["close"]) & data["close"].gt(0) & np.isfinite(previous) & previous.gt(0)
    data["daily_returns"] = ret.where(valid & np.isfinite(ret))
    mean = data.groupby("ticker")["daily_returns"].transform("mean")
    std = data.groupby("ticker")["daily_returns"].transform("std")  # ddof=1
    data["flag_extreme_returns"] = data["daily_returns"].notna() & std.gt(0) & (data["daily_returns"] - mean).abs().gt(3 * std)
    prices = data[["open", "high", "low", "close", "adj_close"]].to_numpy(dtype=float)
    data["flag_bad_prices"] = (np.isinf(prices) | (prices <= 0)).any(axis=1)
    data["flag_impossible_close"] = data["high"].lt(data["close"]) | data["low"].gt(data["close"])
    return data
df = detect_outliers(df)
display(df[["flag_extreme_returns", "flag_bad_prices", "flag_impossible_close"]].sum().to_frame("flagged_rows"))

,flagged_rows
flag_extreme_returns,5056
flag_bad_prices,5
flag_impossible_close,0


In [5]:
# inconsistencies
def detect_inconsistencies(data):
    return pd.DataFrame({
        "invalid_ohlc": data["low"].gt(data["high"]) | data["open"].lt(data["low"]) | data["open"].gt(data["high"]),
        "zero_volume": data["volume"].eq(0),
        "invalid_volume": data["volume"].lt(0) | np.isinf(data["volume"].to_numpy(dtype=float))
    }, index=data.index)
inconsistencies = detect_inconsistencies(df)
display(inconsistencies.sum().to_frame("flagged_rows"))
display(df.loc[df["flag_impossible_close"] | inconsistencies["invalid_ohlc"],["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume"]])

,flagged_rows
invalid_ohlc,5
zero_volume,75
invalid_volume,0


,ticker,ts,open,high,low,close,adj_close,volume
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000


The five nonpositive-price flags and five OHLC-bound violations refer to the same five observations. In each observation, `open=0` while `low>0`. All five have a valid `low`, `high`, and `close`, and no close falls outside its day's high-low range. Zero-volume observations are reported separately and are not treated as proven price errors. The table below shows the existing values and the proposed same-day estimate for `open`; it does not modify the data.

In [6]:
# Display the raw-price errors and a proposed same-day opening-price estimate.
invalid_rows = df["flag_bad_prices"] | df["flag_impossible_close"] | inconsistencies["invalid_ohlc"]
open_review = df.loc[invalid_rows, ["ticker","ts","open","low","high","close","volume"]].copy()
open_review["proposed_open"] = (open_review["low"] + open_review["high"]) / 2
display(open_review)
print("Distinct invalid-price rows:", len(open_review))
print("Close outside low-high:", int(df["flag_impossible_close"].sum()))
print("Zero-volume rows:", int(inconsistencies["zero_volume"].sum()))

,ticker,ts,open,low,high,close,volume,proposed_open
87103,CVX,1962-01-02,0.0,3.24000,3.30000,3.30000,105840,3.270000
167733,JNJ,1962-01-02,0.0,0.22222,0.22338,0.22338,0,0.222800
213412,MCD,1966-07-05,0.0,0.26749,0.27366,0.26955,388800,0.270575
243893,MRK,1962-01-02,0.0,0.39317,0.39703,0.39537,633830,0.395100
287040,PG,1962-01-02,0.0,1.42000,1.44000,1.43000,192000,1.430000


Distinct invalid-price rows: 5
Close outside low-high: 0
Zero-volume rows: 75


In [7]:
# Collect every detected issue into the Data Quality Report.
def create_quality_report(data, gaps, inc):
    checks = {f"missing_{col}": data[col].isna() for col in cols}
    checks.update({
        "extreme_return": data["flag_extreme_returns"],
        "bad_price": data["flag_bad_prices"],
        "impossible_close": data["flag_impossible_close"],
        "invalid_ohlc": inc["invalid_ohlc"],
        "zero_volume": inc["zero_volume"],
        "invalid_volume": inc["invalid_volume"]
    })
    parts = [data.loc[mask, ["ticker", "ts"]].assign(issue_type=name) for name, mask in checks.items()]
    parts += [raw.loc[exact_duplicate, ["ticker", "ts"]].assign(issue_type="exact_duplicate"),gaps.assign(issue_type="missing_date")]
    report = pd.concat(parts, ignore_index=True).rename(columns={"ts": "issue_date"})
    descriptions = {
        "extreme_return": "Raw-close return exceeds 3 sample SDs",
        "bad_price": "Price is nonpositive or infinite",
        "impossible_close": "Close is outside the low-high range",
        "invalid_ohlc": "Open or low-high bounds are invalid",
        "zero_volume": "Zero volume may indicate stale data",
        "invalid_volume": "Volume is negative or infinite",
        "exact_duplicate": "Repeated record after ticker normalization",
        "missing_date": "Reference trading date is absent"
    }
    descriptions.update({f"missing_{col}": f"{col} is missing" for col in cols})
    report["description"] = report["issue_type"].map(descriptions)
    high = {"bad_price", "impossible_close", "invalid_ohlc", "invalid_volume",
            "missing_ts", "missing_ticker", "missing_open", "missing_high", "missing_low", "missing_close"}
    report["severity"] = np.select([report["issue_type"].isin(high), report["issue_type"].isin({"zero_volume", "exact_duplicate"})],["high", "low"], default="medium")
    return report[["ticker", "issue_type", "issue_date", "description", "severity"]].sort_values(["ticker", "issue_date", "issue_type"]).reset_index(drop=True)

quality_report = create_quality_report(df, missing_dates, inconsistencies)
display(quality_report.head(20))
display(quality_report["issue_type"].value_counts().to_frame("issues"))

,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,Raw-close return exceeds 3 sample SDs,medium
1,AAPL,exact_duplicate,1981-01-02,Repeated record after ticker normalization,low
2,AAPL,exact_duplicate,1981-01-05,Repeated record after ticker normalization,low
3,AAPL,missing_adj_close,1981-01-08,adj_close is missing,medium
4,AAPL,missing_adj_close,1981-01-13,adj_close is missing,medium
5,AAPL,exact_duplicate,1981-02-10,Repeated record after ticker normalization,low
6,AAPL,missing_adj_close,1981-02-12,adj_close is missing,medium
7,AAPL,exact_duplicate,1981-02-27,Repeated record after ticker normalization,low
8,AAPL,exact_duplicate,1981-03-05,Repeated record after ticker normalization,low
9,AAPL,exact_duplicate,1981-03-06,Repeated record after ticker normalization,low


,issues
issue_type,
exact_duplicate,29460
missing_adj_close,18374
extreme_return,5056
zero_volume,75
bad_price,5
invalid_ohlc,5


### 1.3 Clean the Data

Only `adj_close` has missing cells in the observed data; raw OHLC prices and volume have no missing cells. For each missing adjusted close, use the supplied `ohlc` row with the same ticker and date. Its factor is `reference adj_close / reference close`, and the reconstructed value is `homework close × factor`. This changes only originally missing adjusted closes. The audit found 18,374 matched rows, no mismatched raw closes, no invalid factors, and no reconstructed values differing from their same-day reference.

In [8]:
# Handle Mssing Values
# since there is no missing values for ohlc & volumes, the only thing to do is to address adj_close
# Reconstruct missing adj_close using only a valid, same-ticker, same-date reference factor
with sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True) as conn:
    ref = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn, parse_dates=["ts"])
# Fill missing adjusted closes with the matched same-day adjustment factor.
ref["factor"] = ref["adj_close"] / ref["close"]
clean = df.merge(ref[["ticker", "ts", "factor"]], on=["ticker", "ts"], how="left", validate="many_to_one")
missing_adj = clean["adj_close"].isna()
clean.loc[missing_adj, "adj_close"] = clean.loc[missing_adj, "close"] * clean.loc[missing_adj, "factor"]
print("Originally missing:", missing_adj.sum())
print("Still missing:", clean["adj_close"].isna().sum())

Originally missing: 18374
Still missing: 0


I choose the assignment's option of estimating a bad price from other valid prices on the same date. For a row with `open=0`, positive finite `low` and `high`, and a valid close inside that range, set `open` to `(low + high) / 2`. This estimate is explicitly marked; it is not claimed to be the actual historical opening trade. No row is removed, and `close`, `adj_close`, and volume are unchanged. The next cell saves the original and estimated opening prices for review.

In [9]:
# Handle Outliers - impossible price relationships
# Estimate only the five invalid opening prices from their same-day range.
def repair_invalid_open(data):
    result = data.copy()
    valid_range = np.isfinite(result[["low","high","close"]]).all(axis=1)
    repair = (result["open"].eq(0) & result["low"].gt(0) & result["high"].ge(result["low"])
              & result["close"].between(result["low"],result["high"]) & valid_range)
    log = result.loc[repair, ["ticker","ts","open","low","high","close"]].copy()
    log = log.rename(columns={"open":"old_open"})
    log["new_open"] = (log["low"] + log["high"]) / 2
    result["flag_open_imputed"] = repair
    result.loc[repair, "open"] = log["new_open"]
    return result, log

clean, open_repairs = repair_invalid_open(clean)
print("Opening prices estimated:", len(open_repairs))
display(open_repairs)

Opening prices estimated: 5


,ticker,ts,old_open,low,high,close,new_open
87103,CVX,1962-01-02,0.0,3.24000,3.30000,3.30000,3.270000
167733,JNJ,1962-01-02,0.0,0.22222,0.22338,0.22338,0.222800
213412,MCD,1966-07-05,0.0,0.26749,0.27366,0.26955,0.270575
243893,MRK,1962-01-02,0.0,0.39317,0.39703,0.39537,0.395100
287040,PG,1962-01-02,0.0,1.42000,1.44000,1.43000,1.430000


The previous missing-value step did not change any originally present `adj_close`. Some of those original values are inconsistent with the supplied same-day `ohlc` reference. For example, GS on 2008-07-16 has the same raw close, 172.86, in both tables, but the homework adjusted close is 3112.52 while the reference adjusted close is 131.50. Compare originally present values with the same-day reference, allowing 0.1% for numeric rounding. Where raw closes agree and the reference adjusted price and factor are valid, recalculate only that row's `adj_close` using its same-day factor. Save old and new values and a repair flag.

In [10]:
# Compare originally present adjusted closes with the matched same-day reference.
original_adj = df[["ticker","ts","adj_close"]].rename(columns={"adj_close":"original_adj"})
ref_prices = ref[["ticker","ts","close","adj_close"]].rename(
    columns={"close":"ref_close","adj_close":"ref_adj"}
)
review = clean.merge(original_adj, on=["ticker","ts"], how="left", validate="one_to_one")
review = review.merge(ref_prices, on=["ticker","ts"], how="left", validate="one_to_one")
same_close = np.isclose(review["close"], review["ref_close"], rtol=1e-6, atol=1e-8)
valid_ref = (same_close & np.isfinite(review["ref_adj"]) & review["ref_adj"].gt(0)
             & np.isfinite(review["factor"]) & review["factor"].gt(0))
different = review["original_adj"].notna() & ~np.isclose(
    review["original_adj"], review["ref_adj"], rtol=1e-3, atol=1e-8
)
review["flag_adj_filled"] = review["original_adj"].isna()
review["flag_adj_repaired"] = different & valid_ref
adj_repairs = review.loc[review["flag_adj_repaired"],
                         ["ticker","ts","original_adj","ref_adj","factor"]].rename(
    columns={"original_adj":"old_adj","ref_adj":"new_adj"}).copy()
review.loc[review["flag_adj_repaired"], "adj_close"] = (
    review.loc[review["flag_adj_repaired"], "close"]
    * review.loc[review["flag_adj_repaired"], "factor"]
)
clean = review.drop(columns=["original_adj","ref_close","ref_adj"]).copy()
print("Originally present adjusted closes repaired:", len(adj_repairs))
display(adj_repairs.head(10))

Originally present adjusted closes repaired: 10417


,ticker,ts,old_adj,new_adj,factor
35,AAPL,1981-02-03,1.264587,0.095069,0.770853
93,AAPL,1981-04-28,2.036653,0.097220,0.770854
101,AAPL,1981-05-08,2.129290,0.096360,0.770878
157,AAPL,1981-07-29,0.506329,0.081734,0.770857
163,AAPL,1981-08-06,2.009828,0.086896,0.770899
167,AAPL,1981-08-13,1.122224,0.080013,0.770841
168,AAPL,1981-08-14,1.171347,0.078722,0.770878
206,AAPL,1981-10-08,1.245215,0.063666,0.770878
264,AAPL,1981-12-31,1.730921,0.076141,0.770878
274,AAPL,1982-01-15,1.577125,0.068829,0.770878


Start with the raw-close 3σ observations. Apply three rules in order:

1. If the same-date adjustment factor changes materially and the adjusted-close return does not exceed the ticker's 3σ threshold, keep the prices and mark `flag_factor_change`.
2. Otherwise, if the adjusted-close return still exceeds 3σ and raw close differs from the same-date `ohlc` reference, mark `flag_data_error`. Estimate adjusted close using valid observations of the same ticker, then recover raw close as `estimated adj_close / same-date factor`. Handle consecutive error rows as one missing run. Record whether the estimate uses observations on both sides or only one side.
3. Keep all remaining raw 3σ observations without another category flag or a price change.

When a recovered close does not fit the original OHLC range, estimate open, high, and low from valid neighboring observations. Do not force an invalid range to pass. Recalculate raw returns after repair because a wrong close can also produce a false return on the next date.

In [11]:
# Classify the original raw-close 3σ observations.
clean = detect_outliers(clean.sort_values(["ticker","ts"]).reset_index(drop=True).copy())
initial_3sigma = int(clean["flag_extreme_returns"].sum())
g = clean.groupby("ticker", sort=False)
adj_return = clean["adj_close"].div(g["adj_close"].shift(1)).sub(1)
adj_mean = adj_return.groupby(clean["ticker"]).transform("mean")
adj_std = adj_return.groupby(clean["ticker"]).transform("std")  # ddof=1
adj_extreme = adj_return.notna() & adj_std.gt(0) & adj_return.sub(adj_mean).abs().gt(3 * adj_std)
prev_factor = g["factor"].shift(1)
factor_changed = prev_factor.notna() & ~np.isclose(clean["factor"],prev_factor,rtol=1e-3,atol=1e-8)
# Reuse ref_prices already prepared in the preceding adjusted-close repair.
clean = clean.merge(ref_prices,on=["ticker","ts"],how="left",validate="one_to_one")
ref_values = clean[["ref_close","ref_adj"]].to_numpy(dtype=float)
ref_valid = ((np.isfinite(ref_values) & (ref_values > 0)).all(axis=1)
             & np.isclose(clean["factor"],clean["ref_adj"].div(clean["ref_close"]),
                          rtol=1e-6,atol=1e-8))
ref_diff = clean["ref_close"].notna() & ~np.isclose(
    clean["close"],clean["ref_close"],rtol=1e-6,atol=1e-8
)
raw_flag = clean["flag_extreme_returns"].copy()
clean["flag_factor_change"] = raw_flag & factor_changed & ~adj_extreme
clean["flag_data_error"] = raw_flag & ~clean["flag_factor_change"] & adj_extreme & ref_diff
# Mask all data-error rows before estimating adjusted close.
bad = clean["flag_data_error"].copy()
invalid_bounds = detect_inconsistencies(clean)["invalid_ohlc"] | clean["flag_impossible_close"]
valid_neighbor = (~bad & ~invalid_bounds & ~clean["flag_bad_prices"]
                  & np.isfinite(clean["adj_close"]) & clean["adj_close"].gt(0))
observed_adj = clean["adj_close"].where(valid_neighbor)
grouped = observed_adj.groupby(clean["ticker"])
two_sided = grouped.transform(lambda s:s.interpolate(method="linear",limit_area="inside"))
one_sided = grouped.ffill().fillna(grouped.bfill())
estimated_adj = two_sided.fillna(one_sided)
new_close = estimated_adj.div(clean["factor"])
# Preserve valid original OHL; otherwise estimate OHL using valid neighbors.
ohl_cols = ["open","high","low"]
observed_ohl = clean[ohl_cols].copy()
observed_ohl.loc[~valid_neighbor,ohl_cols] = np.nan
estimated_ohl = observed_ohl.groupby(clean["ticker"]).transform(
    lambda s:s.interpolate(method="linear",limit_area="inside").ffill().bfill()
)
original_range_ok = (clean["low"].gt(0) & clean["low"].le(clean["open"])
                     & clean["open"].le(clean["high"])
                     & new_close.between(clean["low"],clean["high"]))
estimated_range_ok = (estimated_ohl["low"].gt(0)
                      & estimated_ohl["low"].le(estimated_ohl["open"])
                      & estimated_ohl["open"].le(estimated_ohl["high"])
                      & new_close.between(estimated_ohl["low"],estimated_ohl["high"]))
factor_ok = np.isfinite(clean["factor"]) & clean["factor"].gt(0)
repaired = (bad & ref_valid & factor_ok & np.isfinite(new_close) & new_close.gt(0)
            & (original_range_ok | estimated_range_ok))
replace_ohl = repaired & ~original_range_ok
# Save old and new values in a notebook-only repair log.
repair_log = clean.loc[repaired,["ticker","ts","close","ref_close","adj_close"]].copy()
repair_log["new_close"] = new_close.loc[repaired].to_numpy()
repair_log["new_adj_close"] = estimated_adj.loc[repaired].to_numpy()
repair_log["method"] = np.where(
    two_sided.loc[repaired].notna(),"two-sided interpolation","one-sided fill"
)
repair_log["OHL_interpolated"] = replace_ohl.loc[repaired].to_numpy()
clean.loc[repaired,"adj_close"] = estimated_adj.loc[repaired].to_numpy()
clean.loc[repaired,"close"] = new_close.loc[repaired].to_numpy()
clean.loc[replace_ohl,ohl_cols] = estimated_ohl.loc[replace_ohl,ohl_cols].to_numpy()
unrepaired = bad & ~repaired
clean = clean.drop(columns=["ref_close","ref_adj"])
# Recalculate the raw 3σ flag after changing prices.
clean = detect_outliers(clean)
clean["flag_factor_change"] &= clean["flag_extreme_returns"]
other_3sigma = clean["flag_extreme_returns"] & ~clean["flag_factor_change"] & ~clean["flag_data_error"]
display(pd.Series({
    "initial_raw_3sigma":initial_3sigma,
    "factor_change":int(clean["flag_factor_change"].sum()),
    "data_error_rows_repaired":int(repaired.sum()),
    "other_final_3sigma":int(other_3sigma.sum()),
    "data_error_rows_not_repaired":int(unrepaired.sum()),
    "one_sided_fills":int((repair_log["method"] == "one-sided fill").sum())
},name="rows").to_frame())
display(repair_log)

,rows
initial_raw_3sigma,5056
factor_change,108
data_error_rows_repaired,0
other_final_3sigma,4948
data_error_rows_not_repaired,0
one_sided_fills,0


,ticker,ts,close,ref_close,adj_close,new_close,new_adj_close,method,OHL_interpolated


Validate every retained row after the price repairs. Prices must be finite and positive, raw OHLC bounds must hold, volume must be finite and nonnegative, ticker-date keys must be unique, and adjusted close must agree with the same-date factor. Any `flag_data_error` row that could not be safely repaired prevents saving. Show before/after counts, including zero-volume observations.

In [12]:
# validation
after_inc = detect_inconsistencies(clean)
prices = clean[["open","high","low","close","adj_close"]].to_numpy(dtype=float)
bad_prices = (~np.isfinite(prices) | (prices <= 0)).any(axis=1)
bad_volume = ~np.isfinite(clean["volume"].to_numpy(dtype=float)) | clean["volume"].lt(0).to_numpy()
checks = pd.Series({
    "invalid_price_rows":int(bad_prices.sum()),
    "invalid_OHLC_rows":int((after_inc["invalid_ohlc"] | clean["flag_impossible_close"]).sum()),
    "invalid_volume_rows":int(bad_volume.sum()),
    "missing_keys":int(clean[["ticker","ts"]].isna().any(axis=1).sum()),
    "duplicate_keys":int(clean.duplicated(["ticker","ts"]).sum()),
    "adj_factor_mismatch":int((~np.isclose(
        clean["adj_close"],clean["close"]*clean["factor"],rtol=1e-3,atol=1e-8
    )).sum()),
    "unrepaired_data_errors":int(unrepaired.sum())
},name="remaining_rows")
display(checks.to_frame())
display(pd.DataFrame({
    "before":[len(df),int(df["adj_close"].isna().sum()),
              int(inconsistencies["invalid_ohlc"].sum()),
              int(inconsistencies["zero_volume"].sum()),initial_3sigma],
    "after":[len(clean),int(clean["adj_close"].isna().sum()),
             int((after_inc["invalid_ohlc"] | clean["flag_impossible_close"]).sum()),
             int(after_inc["zero_volume"].sum()),int(clean["flag_extreme_returns"].sum())]
},index=["rows","missing_adj_close","invalid_OHLC","zero_volume","raw_3sigma"]))
print("Exact duplicates removed:",int(exact_duplicate.sum()))
print("Opening prices repaired:",len(open_repairs),
      "adjusted prices changed:",
      int(clean["flag_adj_filled"].sum()+clean["flag_adj_repaired"].sum()))

,remaining_rows
invalid_price_rows,0
invalid_OHLC_rows,0
invalid_volume_rows,0
missing_keys,0
duplicate_keys,0
adj_factor_mismatch,0
unrepaired_data_errors,0


,before,after
rows,368251,368251
missing_adj_close,18374,0
invalid_OHLC,5,0
zero_volume,75,75
raw_3sigma,5056,5056


Exact duplicates removed: 29460
Opening prices repaired: 5 adjusted prices changed: 28791


Write `ohlc_hw_cleaned` only if every validation count is zero. Preserve the original `ohlc_hw` table and store dates in ISO format. The new table contains the eight data columns and exactly two additional columns: `flag_factor_change` and `flag_data_error`. Keep the other diagnostic flags and the repair log in this notebook.

In [14]:
# Save
flags = ["flag_extreme_returns", "flag_bad_prices", "flag_impossible_close",
         "flag_open_imputed", "flag_adj_filled", "flag_adj_repaired",
         "flag_factor_change", "flag_data_error"]
columns = ["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume"] + flags
output = clean[columns].copy()
output["ts"] = output["ts"].dt.strftime("%Y-%m-%d")
output[flags] = output[flags].astype(int)
with sqlite3.connect(db_path) as conn:
    output.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

In [15]:
with sqlite3.connect(db_path) as conn:
    tables = pd.read_sql_query(
        "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name",
        conn)
display(tables)

,name,type
0,market_caps,table
1,ohlc,table
2,ohlc_hw,table
3,ohlc_hw_cleaned,table
